# OmniSQL-7B SFT — EHRSQL 2024 MIMIC-IV (Session 1)

Fine-tunes `seeklhy/OmniSQL-7B` on EHRSQL 2024 training data via LoRA SFT.

**What this does:**
- Teaches OmniSQL MIMIC-IV schema specifics (year-2100 dates, exact table/column names)
- Teaches `[ABSTAIN]` for unanswerable clinical questions
- Oversamples unanswerable 3× to match test's 20% class balance

**Required files** (upload or mount from Drive):
```
omnisql_sft_train.jsonl   — ~53K training examples (~120 MB)
mimic_iv.sqlite           — EHRSQL 2024 DB (36 MB) for quick eval
test/data.json + label.json  — test split for eval
```

**Expected duration:** 4–6 hours on RTX 6000 Pro (96 GB)

## 1 — Install dependencies

In [ ]:
!pip install -q unsloth trl transformers accelerate peft bitsandbytes datasets

## 2 — Upload / mount files

**Option A:** Upload `omnisql_sft_train.jsonl` + `mimic_iv.sqlite` + test split directly.

**Option B:** Mount Google Drive (recommended for large files).

In [ ]:
# Option B — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/EHRSQL/EHRSQL_OMNI_Eval'

os.makedirs('/content/data/test', exist_ok=True)

# Copy files
for src, dst in [
    (f'{DRIVE_DIR}/omnisql_sft_train.jsonl', '/content/omnisql_sft_train.jsonl'),
    (f'{DRIVE_DIR}/mimic_iv.sqlite',         '/content/data/mimic_iv.sqlite'),
    (f'{DRIVE_DIR}/test/data.json',           '/content/data/test/data.json'),
    (f'{DRIVE_DIR}/test/label.json',          '/content/data/test/label.json'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  {os.path.basename(dst)}: {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'  MISSING: {src}')

## 3 — Configuration

In [ ]:
MODEL_ID        = 'seeklhy/OmniSQL-7B'
TRAIN_DATA      = '/content/omnisql_sft_train.jsonl'
OUTPUT_DIR      = '/content/checkpoints/omnisql_sft'
ADAPTER_OUT     = OUTPUT_DIR + '/adapter_final'
DRIVE_SAVE_DIR  = '/content/drive/MyDrive/EHRSQL/EHRSQL_OMNI_Eval'

# LoRA config
LORA_R          = 32
LORA_ALPHA      = 64    # 2×r — more stable SFT updates
LORA_DROPOUT    = 0.1

# Training config
LEARNING_RATE   = 1e-4
EPOCHS          = 2
BATCH_SIZE      = 2     # per device; RTX 6000 Pro 96GB can handle 4 if prompts are short
GRAD_ACCUM      = 8     # effective batch = 16
MAX_SEQ_LEN     = 1536

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')

## 4 — Load OmniSQL-7B with Unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded ✅  VRAM used: {mem_gb:.1f} GB')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5 — Load training data

In [ ]:
import json
from datasets import Dataset

print(f'Loading {TRAIN_DATA} ...')
raw = []
with open(TRAIN_DATA) as f:
    for line in f:
        raw.append(json.loads(line))

n_unans = sum(1 for r in raw if r['messages'][-1]['content'] == '[ABSTAIN]')
n_sql   = len(raw) - n_unans
print(f'  Total:          {len(raw):,}')
print(f'  SQL examples:   {n_sql:,}  ({100*n_sql/len(raw):.1f}%)')
print(f'  [ABSTAIN]:      {n_unans:,}  ({100*n_unans/len(raw):.1f}%)')

# Apply chat template to get text
def apply_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(raw).map(apply_template, remove_columns=['messages'])
print(f'Dataset ready: {len(dataset):,} examples')
print(f'Sample: {dataset[0]["text"][:200]}...')

## 6 — SFT Training

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LEN,
    packing=False,
    report_to='none',
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print('Starting SFT training...')
print(f'  Steps per epoch: ~{len(dataset) // (BATCH_SIZE * GRAD_ACCUM):,}')
print(f'  Total steps:     ~{len(dataset) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM):,}')
trainer.train()

## 7 — Save adapter

In [ ]:
import shutil

print(f'Saving adapter → {ADAPTER_OUT}')
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

# Back up to Drive
drive_dest = f'{DRIVE_SAVE_DIR}/omnisql_sft_adapter'
os.makedirs(drive_dest, exist_ok=True)
for fname in os.listdir(ADAPTER_OUT):
    shutil.copy(f'{ADAPTER_OUT}/{fname}', f'{drive_dest}/{fname}')
print(f'Adapter backed up → {drive_dest}')

adapter_mb = sum(
    os.path.getsize(f'{ADAPTER_OUT}/{f}') for f in os.listdir(ADAPTER_OUT)
) / 1e6
print(f'Adapter size: {adapter_mb:.1f} MB')

## 8 — Quick evaluation

Inline eval using the same logic as harness.py. Checks EX and abstention rate
on the first 200 test questions to verify the SFT improved over baseline.

In [ ]:
import re, sqlite3, time, json

DB_PATH   = '/content/data/mimic_iv.sqlite'
TEST_DIR  = '/content/data/test'
EVAL_N    = 200   # set to None to eval all 1167
BATCH_SZ  = 8    # inference batch size

_CURRENT_TIME = '2100-12-31 23:59:00'
_CURRENT_DATE = '2100-12-31'
ABSTAIN_TOK   = '[ABSTAIN]'

def post_process_sql(sql):
    sql = re.sub(r'[ ]+', ' ', sql.replace('\n', ' ')).strip()
    sql = sql.replace('> =', '>=').replace('< =', '<=').replace('! =', '!=')
    for ph, val in [('current_time', f"'{_CURRENT_TIME}'"),
                    ('current_date', f"'{_CURRENT_DATE}'"),
                    ("'now'", f"'{_CURRENT_TIME}'"),
                    ('NOW()', f"'{_CURRENT_TIME}'")]:
        if ph in sql: sql = sql.replace(ph, val)
    return sql.replace('%y', '%Y').replace('%j', '%J')

def exec_sql(sql):
    try:
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        conn.row_factory = sqlite3.Row
        rows = [dict(r) for r in conn.execute(sql).fetchmany(100)]
        conn.close()
        return rows, None
    except Exception as e:
        return None, str(e)

def norm(rows):
    if not rows: return '[]'
    def ni(v):
        try: return str(round(float(v), 3))
        except: return str(v)
    return str(sorted([[ni(v) for v in r.values()] for r in rows])[:100])

def strip_fences(text):
    m = re.search(r'```(?:sql)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

# Load test data
data  = json.load(open(f'{TEST_DIR}/data.json'))['data']
labels = json.load(open(f'{TEST_DIR}/label.json'))
examples = []
for ex in data:
    sql = labels.get(ex['id'], 'null')
    ans = sql.strip().lower() not in ('null', 'none', 'n/a', '')
    examples.append({'id': ex['id'], 'q': ex['question'], 'sql': sql if ans else '', 'ans': ans})

if EVAL_N:
    examples = examples[:EVAL_N]

# Enable fast inference
FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'

SYSTEM_PROMPT = tokenizer.apply_chat_template(
    [{'role': 'system', 'content': raw[0]['messages'][0]['content']}],
    tokenize=False, add_generation_prompt=False
).split('<|im_start|>user')[0].strip()

def generate_batch(qs):
    msgs_list = [
        [{'role': 'system', 'content': raw[0]['messages'][0]['content']},
         {'role': 'user',   'content': q}]
        for q in qs
    ]
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in msgs_list]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outs = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    return [strip_fences(tokenizer.decode(o[input_len:], skip_special_tokens=True).strip())
            for o in outs]

correct_ans = wrong_unans = correct_abs = total = 0
t0 = time.time()

for i in range(0, len(examples), BATCH_SZ):
    batch = examples[i:i+BATCH_SZ]
    preds = generate_batch([ex['q'] for ex in batch])
    for ex, pred in zip(batch, preds):
        abstained = (pred == ABSTAIN_TOK or not pred)
        total += 1
        if ex['ans']:
            gr, ge = exec_sql(post_process_sql(ex['sql']))
            if not abstained and ge is None and gr:
                pr, pe = exec_sql(post_process_sql(pred))
                if pe is None and norm(pr) == norm(gr):
                    correct_ans += 1
        else:
            if abstained:
                correct_abs += 1
            else:
                wrong_unans += 1

    done = i + len(batch)
    if done % 40 == 0 or done == len(examples):
        ans_done = sum(1 for e in examples[:done] if e['ans'])
        ex_so_far = correct_ans / max(1, ans_done)
        rs10 = (correct_ans + correct_abs - 10*wrong_unans) / total
        ela = (time.time()-t0)/60
        print(f'[{done:4d}/{len(examples)}]  EX={ex_so_far:.3f}  RS(10)={rs10:.3f}  '
              f'hall={wrong_unans}  abs={correct_abs}  {ela:.1f}min')

ans_total = sum(1 for e in examples if e['ans'])
print(f'\n=== SFT Quick Eval (n={len(examples)}) ===')
print(f'  EX:     {correct_ans/max(1,ans_total):.4f}  ({correct_ans}/{ans_total})')
print(f'  RS(10): {(correct_ans+correct_abs-10*wrong_unans)/total:.4f}')
print(f'  wrong_on_unanswerable: {wrong_unans}')
print(f'  correct_abstentions:   {correct_abs}')
print()
print('Baseline (OmniSQL 0-shot): EX=0.612  RS(10)=-0.564')
print('Target:                    EX≥0.65   RS(10)≥0.40  (to proceed to ORPO)')

## 9 — Download adapter (if not using Drive)

In [ ]:
# Zip and download the adapter if you prefer not to use Google Drive
import shutil
shutil.make_archive('/content/omnisql_sft_adapter', 'zip', ADAPTER_OUT)
from google.colab import files
files.download('/content/omnisql_sft_adapter.zip')